# NHF validation

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

##### funcs to get best fit

In [ ]:
def psi_poly(T, c, b, a, e):
    """base transfer function"""
    # return e * x**3 + c * x**2 + b * x + a
    return (c / 3) * T**3 + (b / 2) * T**2 + a * T + e


def get_psi_poly(x, y):
    """get transfer function from data"""

    ## prep data
    if "member" in x.dims:
        stack = lambda x: x.stack(s=["time", "member"]).values
        x = stack(x)
        y = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_poly,
        xdata=x,
        ydata=y,
    )

    ## define transfer func
    psi = lambda T: psi_poly(T, c=p[0], b=p[1], a=p[2], e=p[3])

    ## put params in dataarray
    params = xr.DataArray(p, coords=dict(param=["c", "b", "a", "e"])).rename("params")

    return psi, params

def psi_exp(x, a, b, c):
    """base transfer function"""
    return c * np.exp(a * x) + b


def get_psi_exp(x, y):
    """get transfer function from data"""

    ## prep data
    if "member" in x.dims:
        stack = lambda x: x.stack(s=["time", "member"]).values
        x = stack(x)
        y = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_exp,
        xdata=x,
        ydata=y,
        p0=np.array([0.7, 1.7, 1]),
    )

    ## define transfer func
    psi = lambda x: psi_exp(x, a=p[0], b=p[1], c=p[2])

    ## put parameters in array
    params = xr.DataArray(p, coords=dict(param=["a", "b", "c"])).rename("params")

    return psi, params



def get_psi_wrapper(xy, x_var, y_var, get_psi_func=get_psi_poly):
    """get transfer function from data"""

    ## get transfer func
    psi_func, p = get_psi_func(x=xy[x_var], y=xy[y_var])

    ## get coefficient array (standardized units)
    z = np.arange(-3.5, 3.6, 0.1)
    z = xr.DataArray(z, coords=dict(sigma=z))

    ## evaluate function
    psi_eval = xr.zeros_like(z)
    psi_eval.values = psi_func(z.values)

    ## merge eval and params
    return xr.merge([psi_eval.rename("psi_eval"), p])

## load data

### ERA5

#### heat/radiation fluxes

In [ ]:
## nhf
nhf_era5 = xr.open_dataset(DATA_FP / "era5_nhf_glade.nc").drop_vars("utc_date")

## make sure latitude is ascending
nhf_era5 = nhf_era5.reindex({"latitude": nhf_era5.latitude[::-1]})

## rename flux variables
varname_dict = dict(
    MSLHF="lhf",
    MSSHF="shf",
    MSNLWRF="lw",
    MSNLWRFCS="lw_cs",
    MSNSWRF="sw",
    MSNSWRFCS="sw_cs",
)
nhf_era5 = nhf_era5.rename(varname_dict)

## net heat flux
## Note: "The ECMWF convention for vertical fluxes is positive downwards."
nhf_era5["nhf"] = nhf_era5["sw"] + nhf_era5["lw"] + nhf_era5["lhf"] + nhf_era5["shf"]

#### SST

In [ ]:
## load ERSST
REF_FP = pathlib.Path("/glade/campaign/cgd/cas/asphilli/CVDP-OBS/")

sst_ersst = xr.merge(
    [
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["ssta"],
        xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["sst"],
        # xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"],
    ]
)

sst_ersst = sst_ersst.rename({"lon": "longitude", "lat": "latitude"})
sst_ersst["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")
sst_ersst = sst_ersst.sel(time=nhf_era5.time)

## compute Ttrop
sst_ersst["Ttrop"] = sst_ersst["sst"].sel(latitude=slice(-15,15)).mean(["latitude","longitude"])
sst_ersst["sst_rel"] = sst_ersst["sst"]-sst_ersst["Ttrop"]

## rename
sst_ersst = sst_ersst.rename({"sst":"sst_total", "ssta":"sst"})

In [ ]:
## load precip
pr_ref = xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"]
pr_ref = pr_ref.rename({"lon":"longitude", "lat":"latitude"}).interp(
    {"latitude":sst_ersst.latitude, "longitude":sst_ersst.longitude}
).rename("pr")
    

### CESM2

In [ ]:
## load data
forced, anom = src.utils.load_consolidated()

## load flux data
flux_forced, flux_anom = src.utils.load_flux_data()

## update sign of LHFLX
flux_anom["lhflx"] = -flux_anom["lhflx"]
flux_forced["lhflx"] = -flux_forced["lhflx"]

## merge
anom = xr.merge([anom, flux_anom])
forced = xr.merge([forced, flux_forced])

## subset in time
anom = anom.sel(time=slice("1950", "1980")).compute()
forced = forced.sel(time=slice("1950", "1980")).compute()

## rename flux variables
varname_dict_cesm = dict(
    lhflx="lhf",
    flns="lw",
    fsns="sw",
    fsnsc="sw_cs",
)

## add component information
for v0 in list(varname_dict_cesm.keys()):
    varname_dict_cesm[f"{v0}_comp"] = f"{varname_dict_cesm[v0]}_comp"

## update names
anom = anom.rename(varname_dict_cesm)
forced = forced.rename(varname_dict_cesm)

## subset for given variables
varnames = ["sst", "pr", "nhf", "sw", "lw", "lhf", "sw_cs"]
varnames += [f"{v}_comp" for v in varnames]
anom = anom[varnames]
forced = forced[varnames]

## load tropical SST for CESM2
Ttrop_mod = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))
anom["Ttrop"] = Ttrop_mod["trop_sst_15"].sel(time=anom.time)

## add total SST to anom for convenience
anom["sst_total"] = anom["sst"]+forced["sst"]
anom["sst_total_comp"] = anom["sst_comp"]

## convert precip to mm/day
anom["pr"] = anom["pr"] * 8.6e4
forced["pr"] = forced["pr"] * 8.6e4

### Compute indices

In [ ]:
fns = dict(
    _3=src.utils.get_nino3,
    _34=src.utils.get_nino34,
    _4=src.utils.get_nino4,
    _e=get_Te,
    _w=get_Tw,
)

#### ERA5

In [ ]:
idx_obs = xr.Dataset()
# idx_mod = xr.Dataset()
for ds in [nhf_era5, sst_ersst[["sst","sst_total"]], pr_ref.to_dataset()]:
    for fn_name, fn in tqdm.tqdm(fns.items()):
        for v in list(ds):
            idx_obs[f"{v}{fn_name}"] = fn(ds[v])

## add relative SST
for v in list(idx_obs):
    if "sst_total" in v:
        idx_obs[f"sst_rel_{v[10:]}"] = idx_obs[v]-sst_ersst["Ttrop"]

#### CESM2

In [ ]:
idx_mod = xr.Dataset()
idx_mod_forced = xr.Dataset()

for n, fn in tqdm.tqdm(fns.items()):

    ## compute
    idx_mod_ = src.utils.reconstruct_wrapper(anom, fn)
    idx_mod_forced_ = src.utils.reconstruct_wrapper(forced, fn)

    ## rename
    name_dict = dict(zip(list(idx_mod_), [f"{i}{n}" for i in list(idx_mod_)]))
    idx_mod = xr.merge([idx_mod, idx_mod_.rename(name_dict)])

    name_dict = dict(zip(list(idx_mod_forced_), [f"{i}{n}" for i in list(idx_mod_forced_)]))
    idx_mod_forced = xr.merge([idx_mod_forced, idx_mod_forced_.rename(name_dict)])

## add relative SST
for v in list(idx_mod):
    if "sst_total" in v:
        idx_mod[f"sst_rel_{v[10:]}"] = idx_mod[v]-anom["Ttrop"]

## get total
idx_mod_total = idx_mod+idx_mod_forced

### resample to NDJ

In [ ]:
def resample_ndj(x):
    """resample data to NDJ avg"""

    ## resample and trim
    x = x.resample({"time": "QS-NOV"}).mean().isel(time=slice(4, -4, 4))

    ## remove mean
    return x

idx_obs = resample_ndj(idx_obs)
idx_mod = resample_ndj(idx_mod)
idx_mod_total = resample_ndj(idx_mod_total)

normalized version


In [ ]:
for v in list(idx_obs):
    idx_obs[f"{v}_norm"] = idx_obs[v] / idx_obs[v].std()

for v in list(idx_mod):
    idx_mod[f"{v}_norm"] = idx_mod[v] / idx_mod[v].std()

## Plots

### ERA5

#### Scatter and timeseries

In [ ]:
## useful things
yr = idx_obs.time.dt.year
ax_kwargs = dict(c="k", ls="--", lw=0.8, zorder=0.5)
prep = lambda x : x-x.mean()

for suf in ["e", "w"]:
    print(f"\nLoc: {suf}")

    fig, axs = plt.subplots(2, 5, figsize=(10, 4.5), layout="constrained")

    for j, v in enumerate(["nhf", "sw", "lhf", "lw", "shf"]):

        ## scatter plot
        axs[0, j].scatter(
            prep(idx_obs[f"sst_{suf}"]),
            prep(idx_obs[f"{v}_{suf}"]),
            s=20,
        )
        axs[0, j].set_title(v)
        axs[0, j].axvline(0, **ax_kwargs)
        axs[0, j].axhline(0, **ax_kwargs)
        axs[0, j].set_xticks([-2, 0, 2])
        axs[0, j].set_xlabel("ssta")

        ## timeseries
        axs[1, j].plot(yr, prep(idx_obs[f"{v}_{suf}"]))
        axs[1, j].axhline(0, **ax_kwargs)
        axs[1, j].axvline(1979, **ax_kwargs)
        axs[1, j].set_xticks([1979, 2022])

    src.utils.set_lims(axs[0])
    src.utils.set_lims(axs[1])
    for ax in axs[:, 0]:
        ax.set_ylabel(r"W m$^{-2}$")
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])
    plt.show()

#### SST over time

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 2.25), layout="constrained")

## timeseries
for ax, suf in zip(axs[:2], ["w", "e"]):
    ax.plot(yr, idx_obs[f"sst_{suf}"])
    ax.axhline(0, **ax_kwargs)
    ax.axvline(1979, **ax_kwargs)
    ax.set_xticks([1979, 2022])

## scatter relationship
axs[-1].scatter(idx_obs[f"sst_e"], idx_obs[f"sst_w"], s=15)
axs[-1].axhline(0, **ax_kwargs)
axs[-1].axvline(0, **ax_kwargs)
axs[-1].set_aspect("equal")
zz = np.linspace(-2, 2.5)
# axs[-1].plot(zz, zz, **dict(ax_kwargs, zorder=10))


## format/labels
src.utils.set_lims(axs[:2])
axs[1].set_yticks([])
axs[0].set_title(r"central SST")
axs[1].set_title(r"east SST")
axs[2].set_ylabel(r"central")
axs[2].set_xlabel(r"east")

plt.show()

In [ ]:
## empty array to hold best fit funcs
psi_evals = np.empty(shape=(2), dtype=object)

fig, axs = plt.subplots(1, 2, figsize=(5.5, 2.75), layout="constrained")

## scatter relationship
for j, (ax, idx_, s, ls) in enumerate(zip(axs, [idx_obs, idx_mod], [20,1], ["-","--"])):

    ## scatter data
    ax.scatter(idx_[f"sst_e_norm"], idx_[f"sst_w_norm"], s=s)

    ## get best fit
    psi_eval = get_psi_wrapper(
        idx_, x_var="sst_e_norm", y_var="sst_w_norm"
    )["psi_eval"]
    min_ = idx_["sst_e_norm"].values.min()
    max_ = idx_["sst_e_norm"].values.max()
    psi_evals[j] = psi_eval.sel(sigma=slice(min_, max_))

    ## plot best fit
    ax.plot(psi_evals[j].sigma, psi_evals[j], c="k", ls=ls)

    ## plot axes
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    ax.set_aspect("equal")

## format/labels
src.utils.set_lims(axs)
axs[1].set_yticks([])

## plot obs best fit on model
axs[1].plot(psi_evals[0].sigma, psi_evals[0], c="k")
axs[0].set_title("ERSST")
axs[1].set_title("CESM2")

plt.show()

#### Precip

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(5,4), layout="constrained")

for j, suf in enumerate(["w","e"]):

    ## specify xvar, yvar
    xvar=f"sst_rel_{suf}"
    yvar=f"pr_{suf}"

    ## scatter data
    axs[0, j].scatter(idx_obs[xvar], idx_obs[yvar], s=20)
    axs[1, j].scatter(idx_mod[xvar], idx_mod_total[yvar], s=1, alpha=.5)
    
    ## get best fit
    psi_kwargs = dict(x_var=xvar, y_var=yvar, get_psi_func=get_psi_exp)
    
    ## obs
    psi_eval_obs = get_psi_wrapper(
        idx_obs.isel(time=~np.isnan(idx_obs[yvar])), **psi_kwargs
    )["psi_eval"]
    max_ = idx_obs[xvar].values.max()+.2
    min_ = idx_obs[xvar].values.min()+.2
    psi_eval_obs = psi_eval_obs.sel(sigma=slice(min_, max_))
    
    ## model
    psi_data_mod = xr.merge([idx_mod[xvar], idx_mod_total[yvar]])
    psi_eval_mod = get_psi_wrapper(psi_data_mod, **psi_kwargs)["psi_eval"]
    max_ = psi_data_mod[xvar].values.max()
    psi_eval_mod = psi_eval_mod.sel(sigma=slice(None, max_))
    
    ## plot best fit
    for ax in axs[:,j]:
        ax.plot(psi_eval_obs.sigma, psi_eval_obs, c="k", ls=ls, lw=2)
    axs[1, j].plot(psi_eval_mod.sigma, psi_eval_mod, c="k", lw=2)
    
        ## plot axes
    for ax in axs[:,j]:
        ax.axhline(0, **ax_kwargs)
        ax.axvline(0, **ax_kwargs)
        # ax.set_aspect("equal")
    
    ## format/labels
    src.utils.set_lims(axs)

for ax in axs[:,1]:
    ax.set_yticks([])
for ax in axs[0,:]:
    ax.set_xticks([])

src.utils.set_lims(axs)

for ax, s in zip(axs[:,-1], ["Obs", "CESM2"]):
    ax.text(s=s, x=1, y=.5, transform=ax.transAxes)

axs[0,0].set_title("C. Pac")
axs[0,1].set_title("E. Pac")

plt.show()

### Comparison between ERA5 and CESM2

##### Make plot

In [ ]:
## useful things
yr = idx_obs.time.dt.year
ax_kwargs = dict(c="k", ls="--", lw=0.8, zorder=0.5)

for suf in ["e", "w"]:

    ## print out location
    print(f"\nLoc: {suf}")

    ## empty array to hold psi funcs
    psi_evals = np.empty(shape=(2, 4), dtype=object)

    fig, axs = plt.subplots(2, 4, figsize=(8, 4.5), layout="constrained")

    ## loop thru flux components
    for j, v in enumerate(["nhf", "sw", "lhf", "lw"]):

        ## label
        axs[0, j].set_title(v)
        axs[1, j].set_xlabel("sst")

        ## get vars
        xvar, yvar = f"sst_{suf}_norm", f"{v}_{suf}"

        ## loop thru model/obs
        for i, (idx_, s, ls, a) in enumerate(
            zip([idx_obs, idx_mod], [20, 1], ["-", "--"], [1, 0.5])
        ):

            ## scatter plot
            axs[i, j].scatter(prep(idx_[xvar]), prep(idx_[yvar]), s=s, alpha=a)

            ## get and plot best fit
            psi_eval = get_psi_wrapper(prep(idx_), x_var=xvar, y_var=yvar)["psi_eval"]
            min_ = prep(idx_[xvar]).values.min()
            max_ = prep(idx_[xvar]).values.max()
            psi_evals[i, j] = psi_eval.sel(sigma=slice(min_, max_))
            axs[i, j].plot(psi_evals[i, j].sigma, psi_evals[i, j], c="k", ls=ls)

            ## guidelines
            axs[i, j].axvline(0, **ax_kwargs)
            axs[i, j].axhline(0, **ax_kwargs)
            axs[i, j].set_xticks([-2, 0, 2])

    ## plot obs. fits on model
    for j, ax in enumerate(axs[1]):
        ax.plot(psi_evals[0, j].sigma, psi_evals[0, j], c="k", ls="-")

    ## formatting
    src.utils.set_lims(axs)
    for ax in axs[:, 0]:
        ax.set_ylabel(r"W m$^{-2}$")
    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])

    for ax in axs[0, :]:
        ax.set_xticks([])
    plt.show()

### PDFs

In [ ]:
edges = np.linspace(-5,2.5, 17)

fig, axs = plt.subplots(1, 2, figsize=(6.5,2.5), layout="constrained")

## west/east
for ax, suf in zip(axs, ["w","e"]):

    ## label
    ax.set_title(suf)
    ax.axvline(0, **ax_kwargs)
    ax.set_xlabel("rel. SST (K)")

    ## obs/model
    for idx_, label in zip([idx_obs, idx_mod], ["ERSST", "CESM2"]):

        ## compute PDF
        pdf, _ = src.utils.get_empirical_pdf(
            idx_[f"sst_rel_{suf}"].values.flatten(), edges=edges,
            # idx_[f"ssta_{suf}"].values.flatten(), edges=edges,
        )

        ax.stairs(pdf, edges, label=label)

src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[0].set_ylabel(r"Density (K$^{-1}$)")
axs[0].legend()
plt.show()

### composites

#### Merge data

In [ ]:
def trim(x):
    return x.sel(latitude=slice(-5,5),longitude=slice(130,280)).mean("latitude")

def prep(x):
    return resample_ndj(trim(x))

#### ERA5

In [ ]:
## new data array to hold trimmed data
data_obs = prep(sst_ersst)

## add heat flux info
data_obs = xr.merge(
    [
        data_obs,
        prep(nhf_era5).interp({"longitude":data_obs.longitude}),
        prep(pr_ref),
        idx_obs,
    ]
)

## compute deviations from mean for convenience
for v in (list(nhf_era5)+list(sst_ersst) + ["pr"]):
    data_obs[f"{v}_anom"] = data_obs[v] - data_obs[v].mean("time")

#### CESM2

In [ ]:
def resample_ndj_cesm(x):
    """resample CESM eofs to djf"""
    comps, scores = src.utils.split_components(x)
    scores_ndj = resample_ndj(scores)
    x_ndj = src.utils.combine_components(comps, scores_ndj)

    return x_ndj

In [ ]:
anom_ndj = resample_ndj_cesm(trim(anom))
data_mod_forced = resample_ndj_cesm(trim(forced))

## create new dataarray
data_mod = xr.merge([anom_ndj, idx_mod])

#### ERA5

In [ ]:
def get_composite(data, idx_name, is_warm=True, n=7):
    """ helper func to make composite"""

    ## get sorted indiex
    idx_sorted = data[idx_name].argsort().values
    if is_warm:
        idx = idx_sorted[-n:]
    else:
        idx = idx_sorted[:n]

    ## select events
    spag = data.isel(time=idx)

    ## rename coordinate
    spag = spag.rename({"time":"sample"}).assign_coords({"sample":np.arange(n)})

    ## rename coordinate
    return spag

In [ ]:
obs_nino_spag = get_composite(data_obs, idx_name="sst_3")
obs_nina_spag = get_composite(data_obs, idx_name="sst_3", is_warm=False)

In [ ]:
fig,axs = plt.subplots(2,5,figsize=(12,4.5))

# ax.plot(obs_nino_spag.longitude, sel(sst_ersst["sst"]).mean("time"), c="k")

## loop thru variables
for ax, varname in zip(axs[0], ["sst_total", "pr", "nhf", "sw", "lhf"]):

    ## plot climatology
    ax.plot(data_obs.longitude, data_obs[varname].mean("time"), c="k")

    ## loop thru warm/cold
    for spags, color in zip([obs_nino_spag, obs_nina_spag], ["r","b"]):

        ax.plot(spags.longitude, spags[varname].mean("sample"), c=color)

## loop thru variables
for ax, varname in zip(axs[1], ["sst", "pr_anom", "nhf_anom", "sw_anom", "lhf_anom"]):

    ## loop thru warm/cold
    for spags, color in zip([obs_nino_spag, -obs_nina_spag], ["r","b"]):

        ## plot
        ax.plot(spags.longitude, spags[varname].mean("sample"), c=color)


## formatting
axs[0,0].set_title("SST")
axs[0,1].set_title("Precip.")
axs[0,-3].set_title("NHF")
axs[0,-2].set_title("Shortwave")
axs[0,-1].set_title("Latent heat flux")
for ax in axs[1]:
    ax.axhline(0, **ax_kwargs)
for ax, s in zip(axs[:,-1], ["Total", "Anom."]):
    ax.text(s=s, x=1.1, y=.5, transform=ax.transAxes)
src.utils.add_vticks(axs.flatten(), xticks=[180,230, 280],xlines=[180, 230])
src.utils.set_lims(axs[1,-3:])
axs[0,-3].set_ylim([0,130])
axs[0,-2].set_ylim([175,275])
axs[0,-1].set_ylim([-150,-50])
for ax in axs[0]:
    ax.set_xticks([])
for ax in axs[1,-3:]:
    ax.set_ylim(ax.get_ylim()[::-1])

plt.show()

#### CESM2

In [ ]:
def get_composite_cesm(data, idx_name, is_warm=True, n=301):
    """ helper func to make composite"""

    ## create sample dim
    data_ = data.stack(sample=["time","member"])

    ## get sorted inde
    idx_sorted = data_[idx_name].argsort().values
    if is_warm:
        idx = idx_sorted[-n:]
    else:
        idx = idx_sorted[:n]

    ## select events
    spag = data_.isel(sample=idx)

    # ## rename coordinate
    # spag = spag.rename({"time":"sample"}).assign_coords({"sample":np.arange(n)})

    ## rename coordinate
    return spag

In [ ]:
## get climatology
mod_clim = src.utils.reconstruct_wrapper(
    data_mod_forced.mean(["time"]),
    fn=lambda x : x,
)

# get_composite_cesm(data_mod, idx_name="ssta_34")
data_mod_components, data_mod_scores = src.utils.split_components(data_mod)

In [ ]:
mod_nino_spag = src.utils.reconstruct_fn(
    scores=get_composite_cesm(data_mod_scores, "sst_34"),
    components=data_mod_components,
    fn=lambda x : x,
)

mod_nina_spag = src.utils.reconstruct_fn(
    scores=get_composite_cesm(data_mod_scores, "sst_34", is_warm=False),
    components=data_mod_components,
    fn=lambda x : x,
)

In [ ]:
fig,axs = plt.subplots(2,5,figsize=(12,4.5), layout="constrained")

# ax.plot(obs_nino_spag.longitude, sel(sst_ersst["sst"]).mean("time"), c="k")

## loop thru variables
for ax, varname in zip(axs[0], ["sst", "pr", "nhf", "sw", "lhf"]):

    ## plot climatology
    ax.plot(mod_clim.longitude, mod_clim[varname], c="k")

    ## loop thru warm/cold
    for spags, color in zip([mod_nino_spag, mod_nina_spag], ["r","b"]):

        ax.plot(spags.longitude, mod_clim[varname]+spags[varname].mean("sample"), c=color)

## loop thru variables
for ax, varname in zip(axs[1], ["sst", "pr", "nhf", "sw", "lhf"]):

    ## loop thru warm/cold
    for spags, color in zip([mod_nino_spag, -mod_nina_spag], ["r","b"]):

        ## plot
        ax.plot(spags.longitude, spags[varname].mean("sample"), c=color)


## formatting
axs[0,0].set_title("SST")
axs[0,1].set_title("Precip")
axs[0,2].set_title("NHF")
axs[0,3].set_title("Shortwave")
axs[0,4].set_title("Latent heat flux")
for ax in axs[1]:
    ax.axhline(0, **ax_kwargs)
for ax, s in zip(axs[:,-1], ["Total", "Anom."]):
    ax.text(s=s, x=1.1, y=.5, transform=ax.transAxes)
src.utils.add_vticks(axs.flatten(), xticks=[180,230, 280],xlines=[180, 230])
src.utils.set_lims(axs[1,2:])
# axs[0,1].set_ylim([175,275])
# axs[0,2].set_ylim([-150,-50])
for ax in axs[0]:
    ax.set_xticks([])
for ax in axs[1,2:]:
    ax.set_ylim(ax.get_ylim()[::-1])

plt.show()

Add precip to this!

## to-do
- same plots, but for CESM2
- relative SST comparison between ERSST and CESM2 (spatial plot?)
- rainfall comparison?